# What is the relationship between each stock's return and a factor value? How should we define the "effectiveness" of a factor?

# Do you still remember our stock-price prediction discussion in Part 1?
<!-- We used many indicators there, but we did not actually validate whether those indicators were effective. In this part, we start doing that validation. -->



---
### Factor in Practice Part 5
# Build Your Own Hedge Fund: Implementing Advanced Hedging Strategies in Python (3)
## Understanding the Statistical Interpretability of Factors

### 🎬 @Director Harold
### 🏛 CUHK Financial Engineering Undergraduate Program
### 📈 On the path to a U.S. Master's in Financial Engineering (already admitted)
### 🌐 [Follow my Bilibili channel for quant learning content that is easy to follow and actually useful](https://space.bilibili.com/629573485)

🌟🌟🌟 Let's pull back the curtain on quantitative investing. #HaroldQuantChannel 🌟

---


Load data


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime

# Use a representative subset of S&P 500 stocks
sp500_tickers = [
    'AAPL', 'MSFT', 'AMZN', 'GOOGL', 'META', 'TSLA', 'BRK-B', 'JNJ', 'V', 'WMT',
    'JPM', 'PG', 'MA', 'UNH', 'HD', 'DIS', 'BAC', 'NVDA', 'ADBE', 'CRM',
    'NFLX', 'PYPL', 'INTC', 'VZ', 'T', 'MRK', 'PFE', 'ABT', 'TMO', 'NKE',
    'XOM', 'CVX', 'LLY', 'MCD', 'KO', 'PEP', 'ABBV', 'AVGO', 'COST', 'ACN',
    'TXN', 'DHR', 'NEE', 'UPS', 'PM', 'LIN', 'LOW', 'QCOM', 'SPGI', 'IBM',
    'BMY', 'AMGN', 'GS', 'BLK', 'CAT', 'GE', 'MMM', 'AXP', 'SCHW', 'BKNG'
]

start = '2005-01-01'
end = '2022-12-31'

monthly_data = []
for ticker in sp500_tickers:
    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(start=start, end=end, interval='1mo')
        hist = hist.reset_index()
        hist['stock'] = ticker
        hist['date'] = hist['Date'].dt.strftime('%Y-%m')
        hist['monthly_return'] = hist['Close'].pct_change()
        info = stock.info
        hist['market_cap'] = hist['Close'] * info.get('sharesOutstanding', 1e9)
        hist['value_in_thousand'] = hist['market_cap'] / 1000
        hist['next_month_return'] = hist['monthly_return'].shift(-1)
        monthly_data.append(hist[['stock', 'date', 'value_in_thousand', 'monthly_return', 'next_month_return']])
    except:
        pass

df_mkt = pd.concat(monthly_data, axis=0).dropna(subset=['monthly_return']).reset_index(drop=True)
unique_dates = df_mkt['date'].unique()
unique_dates.sort()
unique_dates_nochange = unique_dates.copy()
unique_dates = [datetime.strptime(date, "%Y-%m") for date in unique_dates]
df_mkt

SMB factor values


In [ ]:
len(hedge_returns)

In [ ]:
hedge_returns

---

# Factors...?

Harold, didn't you say that a factor can help determine whether a stock goes up or down?

---


Harold two-factor model


In [ ]:
SMB_monthly = pd.DataFrame(hedge_returns, unique_dates_nochange,columns=['SMB_monthly'])

pd.DataFrame(hedge_returns, unique_dates_nochange,columns=['SMB_monthly'])

In [ ]:
df_mkt

In [ ]:
MKT = df_mkt.groupby('date')['next_month_return'].mean()
MKT_monthly = pd.DataFrame(MKT)
MKT_monthly

Now we have monthly values for the MKT factor and the SMB factor.
### For any stock, its monthly return can be decomposed as R = A1 * MKT factor + A2 * SMB factor + alpha


In [ ]:
test_stock_code = 'JPM' # JPMorgan Chase
test_stock = df_mkt[df_mkt['stock'] == test_stock_code]
test_stock

In [ ]:
test_stock_list = df_mkt['stock'].unique()
len(test_stock_list)

In [ ]:
y = test_stock[['date','next_month_return']].set_index('date')
y.dropna(inplace=True)
y

In [ ]:
X = pd.concat([MKT_monthly['next_month_return'], SMB_monthly['SMB_monthly']], axis=1)
X.dropna(inplace=True)
X

In [ ]:
import statsmodels.api as sm
from statsmodels import regression

data_for_stock = y.index.unique()
X = X.loc[data_for_stock]
X = sm.add_constant(X)
model = regression.linear_model.OLS(y, X).fit()
alpha = model.params[0]
beta1 = model.params[1]
beta2 = model.params[2]
print('alpha: ' + str(alpha))
print('beta SMB: ' + str(beta1))
print('beta MKT: ' + str(beta2))
print(model.summary())

In [ ]:
R2 = []
X = pd.concat([MKT_monthly['next_month_return'], SMB_monthly['SMB_monthly']], axis=1)
X.dropna(inplace=True)

for stock in test_stock_list:
    y = df_mkt[df_mkt['stock'] == stock][['date','next_month_return']].set_index('date')
    y.dropna(inplace=True)
    
    if y.empty:
        continue

    data_for_stock = y.index.unique()
    X_stock = X.loc[data_for_stock]  # Use X_stock instead of X
    X_stock = sm.add_constant(X_stock)

    model = regression.linear_model.OLS(y, X_stock).fit()  # Use X_stock instead of X
    R2.append(model.rsquared)

In [ ]:
R2_df = pd.DataFrame(R2).dropna()
R2_df.dropna(inplace=True)
R2_df.describe()

---

# Three factors, and then more...

---


In [ ]:
# Load PB ratio data via yfinance for S&P 500 stocks
pb_data = []
for ticker in sp500_tickers:
    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(start='2005-01-01', end='2022-12-31', interval='1mo')
        hist = hist.reset_index()
        hist['stock'] = ticker + '.US'
        info = stock.info
        name = info.get('shortName', ticker)
        hist['name'] = name
        hist['date'] = hist['Date']
        hist['return'] = hist['Close'].pct_change() * 100
        hist['PB'] = info.get('priceToBook', np.nan)
        hist['next_month_return'] = hist['return'].shift(-1)
        pb_data.append(hist[['stock', 'name', 'date', 'return', 'PB', 'next_month_return']])
    except:
        pass

df_pb = pd.concat(pb_data, axis=0).reset_index(drop=True)
df_pb

In [ ]:
df = df_pb.copy()
df['book_to_market'] = 1 / df['PB']
df

# TODO: build an investment portfolio using the book-to-market ratio


# How can we replicate the Fama-French three-factor model in the US market?

### Reference: Fama + French (1993)
